<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-03-from-notebook-to-reproducible-experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 (graded) — From notebook to reproducible experiment
**Course 1: Hands-On Deep Learning with Python — Chapter 3: PyTorch fluency & the training loop**

**Problem brief (Leo Farkas, Orbit Retail):** "We lose subscribers we could have saved.
Predict who will churn next month so retention can call them first."
Target: recall@top-10%-risk, plus a calibrated probability.

**What you'll submit:** a PyTorch churn model, 5+ MLflow-logged runs with varied
hyperparameters, a comparison table, the recall target hit, and the best run registered.

In [ ]:
!pip install -q mlflow

## 1. Load the data (with offline fallback)

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd

np.random.seed(0)

def load_churn_data():
    try:
        # fetch with a bounded timeout first - pd.read_csv(url) has no timeout of its own
        # and can hang the whole cell indefinitely on a stalled connection
        req = urllib.request.Request(
            'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/'
            'master/data/Telco-Customer-Churn.csv',
            headers={'User-Agent': 'aibits-course-lab/1.0'},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw = resp.read()
        df = pd.read_csv(io.BytesIO(raw))
        print('Loaded the real Telco Customer Churn dataset:', df.shape)
        return df
    except Exception as e:
        print(f'Offline fallback engaged ({e}).')
        n = 3000
        tenure = np.random.randint(0, 72, n)
        monthly = np.random.uniform(18, 120, n)
        contract = np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.55, 0.25, 0.2])
        risk = 1 / (1 + np.exp(-(2.0 - 0.04 * tenure - 0.01 * monthly + (contract == 'Month-to-month') * 1.2)))
        churn = np.where(np.random.rand(n) < risk, 'Yes', 'No')
        return pd.DataFrame({'tenure': tenure, 'MonthlyCharges': monthly,
                              'Contract': contract, 'Churn': churn})

df = load_churn_data()
df = df.copy()
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df = df.dropna(subset=['MonthlyCharges'])

num_cols = [c for c in ['tenure', 'MonthlyCharges'] if c in df.columns]
cat_cols = [c for c in ['Contract'] if c in df.columns]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
feature_cols = num_cols + [c for c in df.columns if c.startswith('Contract_')]

X = df[feature_cols].to_numpy(dtype=np.float32)
X = (X - X.mean(0)) / (X.std(0) + 1e-8)
y = df['Churn'].to_numpy(dtype=np.float32)

n = len(X); idx = np.random.permutation(n); n_train = int(n * 0.75)
X_train, y_train = X[idx[:n_train]], y[idx[:n_train]]
X_val, y_val = X[idx[n_train:]], y[idx[n_train:]]
print('features:', feature_cols)
print('train:', X_train.shape, 'val:', X_val.shape, 'churn rate:', y.mean().round(3))

## 2. Dataset / DataLoader / model / training loop

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class ChurnDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class ChurnMLP(nn.Module):
    def __init__(self, n_in, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),  # logits — BCEWithLogitsLoss applies the sigmoid
        )

    def forward(self, x):
        return self.net(x)


def train_one_run(lr, hidden, weight_decay, n_epochs, batch_size, seed=0):
    torch.manual_seed(seed)
    train_ds = ChurnDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = ChurnMLP(X_train.shape[1], hidden=hidden).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(torch.tensor(X_val, dtype=torch.float32).to(device)).cpu().numpy().ravel()
    val_prob = 1 / (1 + np.exp(-val_logits))
    return model, val_prob

## 3. Metrics: AUC + recall@top-10%-risk

In [ ]:
from sklearn.metrics import roc_auc_score

def recall_at_top_k_pct(y_true, y_prob, pct=0.10):
    k = max(1, int(len(y_prob) * pct))
    top_k_idx = np.argsort(-y_prob)[:k]
    caught = y_true[top_k_idx].sum()
    total_positive = y_true.sum()
    return caught / total_positive if total_positive > 0 else float('nan')

## 4. Run 5+ hyperparameter variants, logged to MLflow

In [ ]:
import mlflow

mlflow.set_experiment('orbit-retail-churn')

configs = [
    {'lr': 1e-3, 'hidden': 16, 'weight_decay': 0.0,   'n_epochs': 20, 'batch_size': 64},
    {'lr': 1e-3, 'hidden': 32, 'weight_decay': 1e-4,  'n_epochs': 20, 'batch_size': 64},
    {'lr': 3e-3, 'hidden': 32, 'weight_decay': 1e-4,  'n_epochs': 20, 'batch_size': 32},
    {'lr': 1e-4, 'hidden': 64, 'weight_decay': 1e-3,  'n_epochs': 30, 'batch_size': 64},
    {'lr': 1e-3, 'hidden': 64, 'weight_decay': 1e-5,  'n_epochs': 30, 'batch_size': 128},
]

results = []
best_auc, best_model, best_run_id = -1, None, None

for i, cfg in enumerate(configs):
    with mlflow.start_run(run_name=f'run-{i}') as run:
        mlflow.log_params(cfg)
        model, val_prob = train_one_run(**cfg)
        auc = roc_auc_score(y_val, val_prob)
        recall10 = recall_at_top_k_pct(y_val, val_prob, pct=0.10)
        mlflow.log_metrics({'val_auc': auc, 'val_recall_at_10pct': recall10})
        mlflow.pytorch.log_model(model, 'model', input_example=X_train[:1], serialization_format='pickle')
        results.append({**cfg, 'val_auc': auc, 'val_recall_at_10pct': recall10, 'run_id': run.info.run_id})
        if auc > best_auc:
            best_auc, best_model, best_run_id = auc, model, run.info.run_id
        print(f'run {i}: AUC={auc:.4f}  recall@10%={recall10:.4f}')

results_df = pd.DataFrame(results)
results_df

## 5. Register the best model

In [ ]:
model_uri = f'runs:/{best_run_id}/model'
registered = mlflow.register_model(model_uri, 'orbit-retail-churn-mlp')
print(f'Registered {registered.name} version {registered.version} from run {best_run_id}')
print(f'Best AUC: {best_auc:.4f}')
print('\nInspect all runs: `!mlflow ui` in a terminal, or read `results_df` above.')

## 6. Comparison table + write-up
Use `results_df` above as your comparison table. In 2–3 sentences: which hyperparameter
mattered most, and does the best run's recall@10% meet retention's operational need?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 3: PyTorch fluency & the training loop*